## thư viện sử dụng

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

## đọc file đã làm sạch

In [2]:
# Tìm thư mục project khi chạy từ notebook hoặc thư mục gốc repo
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/processed").is_dir() or (p / "project/data/processed").is_dir())
if not (ROOT / "data/processed").is_dir():
    ROOT = ROOT / "project"

path = ROOT / "data/processed/data_clean.csv"
# File hiện tại có đuôi .csv nhưng nội dung được nén gzip
with path.open("rb") as f:
    compression = "gzip" if f.read(2) == b"\x1f\x8b" else None

df = pd.read_csv(path, compression=compression,
                 dtype={"Customer ID": "string", "Invoice": "string", "StockCode": "string"},
                 parse_dates=["InvoiceDate"])
print("Dữ liệu sạch:", df.shape)
df.head()

Dữ liệu sạch: (522388, 10)


## chọn khách hàng có mã
Không gom các giao dịch Guest thành một khách hàng. Giữ nguyên quy tắc làm sạch đã có;
không loại thêm ngoại lệ hoặc dòng trùng. Giá trị mua được tính lại bằng Quantity × Price.

In [3]:
ids = df["Customer ID"].str.strip()
guest_flag = df["IsGuest"].astype("string").str.lower()
assert guest_flag.isin(["true", "false"]).all(), "IsGuest không hợp lệ"
guest = guest_flag.eq("true") | ids.isna() | ids.str.lower().isin(["", "guest"])
data = df.loc[~guest].copy()
data["Customer ID"] = ids.loc[~guest].str.replace(r"(?<=\d)\.0$", "", regex=True)
assert not data.empty
assert data[["Customer ID", "Invoice", "StockCode", "InvoiceDate"]].notna().all().all()
assert (data["Quantity"] > 0).all() and (data["Price"] > 0).all()
assert not data["Invoice"].str.upper().str.startswith("C").any()
assert data.groupby("Invoice")["Customer ID"].nunique().max() == 1

data["LineValue"] = data["Quantity"] * data["Price"]
assert np.isfinite(data[["Quantity", "Price", "LineValue"]]).all().all()
reference_date = df["InvoiceDate"].max().normalize() + pd.Timedelta(days=1)
print("Dòng Guest không đưa vào feature:", int(guest.sum()))
print("Ngày tham chiếu:", reference_date)
print("Số khách hàng:", data["Customer ID"].nunique())

Dòng Guest không đưa vào feature: 131401
Ngày tham chiếu: 2011-12-10 00:00:00
Số khách hàng: 4327


## tạo RFM
- Recency: số ngày lịch từ lần mua cuối đến ngày tham chiếu chung (ngày kế tiếp ngày cuối dữ liệu).
- Frequency: số hóa đơn khác nhau, không phải số dòng hàng.
- Monetary: tổng Quantity × Price (GBP), chưa đối soát hoàn trả.

In [4]:
feature = data.groupby("Customer ID").agg(
    LastPurchase=("InvoiceDate", "max"),
    Frequency=("Invoice", "nunique"),
    Monetary=("LineValue", "sum")
)
feature["Recency"] = (reference_date - feature["LastPurchase"].dt.normalize()).dt.days
feature = feature[["Recency", "Frequency", "Monetary"]]
feature.head()

## thêm feature được đề xuất trong EDA
Giá trị trung bình đơn, số mã hàng, tổng lượng, lượng trung bình/đơn, số ngày mua,
khoảng hoạt động, thời gian từ lần đầu mua và khoảng cách mua trung bình.
Số mã/lượng là các mã quan sát trong dữ liệu sạch, chưa khẳng định danh mục sản phẩm đã xác minh.
Không thêm bản sao Frequency/Monetary. Không tính tỷ lệ trả hàng vì dữ liệu sạch đã loại C/âm.

In [5]:
data["PurchaseDay"] = data["InvoiceDate"].dt.normalize()
behavior = data.groupby("Customer ID").agg(
    n_stock_codes=("StockCode", "nunique"),
    total_quantity=("Quantity", "sum"),
    active_days=("PurchaseDay", "nunique"),
    first_purchase=("InvoiceDate", "min"),
    last_purchase=("InvoiceDate", "max")
)
behavior["observed_span_days"] = (behavior["last_purchase"] - behavior["first_purchase"]).dt.total_seconds() / 86400
behavior["observed_tenure_days"] = (reference_date - behavior["first_purchase"]).dt.total_seconds() / 86400

# Mỗi hóa đơn lấy thời điểm sớm nhất, sau đó sắp xếp để tính gap
orders = data.groupby(["Customer ID", "Invoice"], as_index=False)["InvoiceDate"].min()
orders = orders.sort_values(["Customer ID", "InvoiceDate", "Invoice"])
orders["gap_days"] = orders.groupby("Customer ID")["InvoiceDate"].diff().dt.total_seconds() / 86400
behavior["avg_purchase_gap_days"] = orders.groupby("Customer ID")["gap_days"].mean()

feature = feature.join(behavior.drop(columns=["first_purchase", "last_purchase"]))
feature["avg_order_value"] = feature["Monetary"] / feature["Frequency"]
feature["avg_quantity_per_order"] = feature["total_quantity"] / feature["Frequency"]
feature = feature.rename_axis("CustomerID").reset_index()
feature.head()

## kiểm tra kết quả
Khách chỉ một hóa đơn có gap là NaN; không điền 0 vì 0 có nghĩa hai hóa đơn cùng thời điểm.
CustomerID chỉ là khóa, không dùng làm biến phân cụm. Các feature chưa log hoặc scale.

In [6]:
assert feature["CustomerID"].is_unique
assert len(feature) == data["Customer ID"].nunique()
assert np.isclose(feature["Monetary"].sum(), data["LineValue"].sum())
assert feature["Frequency"].sum() == len(orders)
assert np.allclose(feature["avg_order_value"] * feature["Frequency"], feature["Monetary"])
assert feature.loc[feature["Frequency"] == 1, "avg_purchase_gap_days"].isna().all()
assert feature.loc[feature["Frequency"] > 1, "avg_purchase_gap_days"].notna().all()
assert np.isfinite(feature.drop(columns=["CustomerID", "avg_purchase_gap_days"])).all().all()
print("Kích thước feature:", feature.shape)
print("Giá trị thiếu theo cột:")
print(feature.isna().sum())
feature.describe()

Kích thước feature: (4327, 12)
Giá trị thiếu theo cột:
CustomerID                   0
Recency                      0
Frequency                    0
Monetary                     0
n_stock_codes                0
total_quantity               0
active_days                  0
observed_span_days           0
observed_tenure_days         0
avg_purchase_gap_days     1504
avg_order_value              0
avg_quantity_per_order       0
dtype: int64


## lưu file vào data/feature

In [7]:
output_dir = ROOT / "data/feature"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "customer_features.csv"
feature.to_csv(output_path, index=False, encoding="utf-8-sig")

# Đọc lại để kiểm tra file đã lưu
saved = pd.read_csv(output_path, dtype={"CustomerID": "string"})
pd.testing.assert_frame_equal(feature, saved, check_dtype=False)
print("Đã lưu:", output_path)
print("Ngày tham chiếu RFM:", reference_date)
saved.head()

Đã lưu: D:\DOAN\SGU-ML-High-Value-Customer-Prediction\project\data\feature\customer_features.csv
Ngày tham chiếu RFM: 2011-12-10 00:00:00
